# RoPE — RoFormer: Rotary Position Embedding (2021)

_Papers · Transformers & LLMs_

**Rotate each query and key vector by an angle proportional to its position, so attention scores depend only on the relative distance between two tokens — never their absolute positions.**

---

This notebook is a practice scaffold for the **RoPE — RoFormer: Rotary Position Embedding (2021)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn.functional as F

torch.manual_seed(0)

# ---------- Step 1: RoPE in 2-D, from scratch (Eq. 13) ----------
def rope_2d(v, pos, theta=1.0):
    """Rotate a 2-D vector v=[v0,v1] by angle pos*theta. Top row [cos,-sin], bottom [sin,cos]."""
    a = pos * theta
    c, s = torch.cos(a), torch.sin(a)
    v0, v1 = v[0], v[1]
    return torch.stack([c*v0 - s*v1,   # note the MINUS sign (top row)
                        s*v0 + c*v1])

# ---- THE ORACLE: q_m . k_n depends ONLY on (m - n) ----
q = torch.tensor([1.0, 0.0])
k = torch.tensor([0.0, 1.0])
theta = torch.tensor(1.0)

# many (m, n) pairs that all share the SAME difference m - n = -3
pairs = [(0,3), (2,5), (7,10), (100,103), (1,4)]
scores = []
for m, n in pairs:
    qm = rope_2d(q, torch.tensor(float(m)), theta)
    kn = rope_2d(k, torch.tensor(float(n)), theta)
    scores.append(torch.dot(qm, kn))
scores = torch.stack(scores)
print("scores for m-n=-3 pairs:", [round(x.item(), 6) for x in scores])
same = torch.allclose(scores, scores[0].expand_as(scores), atol=1e-6)
print("all equal (depends only on m-n)?", same)        # expect True  -> ~ -0.141120

# a DIFFERENT difference gives a DIFFERENT (but internally consistent) score
sc2 = torch.dot(rope_2d(q, torch.tensor(0.), theta), rope_2d(k, torch.tensor(1.), theta))
print("score for m-n=-1:", round(sc2.item(), 6), "(differs from -0.1411, as it should)")

# recompute the WORKED EXAMPLE numbers exactly
print("q_2 =", [round(x,4) for x in rope_2d(q, torch.tensor(2.), theta).tolist()])  # [-0.4161, 0.9093]
print("k_5 =", [round(x,4) for x in rope_2d(k, torch.tensor(5.), theta).tolist()])  # [ 0.9589, 0.2837]

# ---------- Step 2: general d-dim RoPE via the elementwise trick (Eq. 34) ----------
def rope(x, pos, base=10000.0):
    """x: (..., d) with d even. pos: scalar position. Pairs adjacent coords (Eq. 15)."""
    d = x.shape[-1]
    i = torch.arange(0, d, 2, dtype=x.dtype)           # 0,2,4,...
    theta = base ** (-(i) / d)                          # theta_i = 10000^{-2(i-1)/d}
    ang = pos * theta                                   # angle per pair
    cos = torch.repeat_interleave(torch.cos(ang), 2)   # [c1,c1,c2,c2,...]
    sin = torch.repeat_interleave(torch.sin(ang), 2)
    x1, x2 = x[..., 0::2], x[..., 1::2]
    swap = torch.stack([-x2, x1], dim=-1).reshape(x.shape)  # [-x2,x1,-x4,x3,...]
    return x * cos + swap * sin

# verify the relative property in d dimensions too
qd = torch.randn(8); kd = torch.randn(8)
s_a = torch.dot(rope(qd, torch.tensor(2.)), rope(kd, torch.tensor(5.)))   # m-n=-3
s_b = torch.dot(rope(qd, torch.tensor(9.)), rope(kd, torch.tensor(12.)))  # m-n=-3
print("d=8 relative property holds:", torch.allclose(s_a, s_b, atol=1e-5))  # expect True

# ---------- Step 3: use RoPE inside a tiny scaled dot-product attention ----------
T, d = 4, 8
Q = torch.randn(T, d); K = torch.randn(T, d); V = torch.randn(T, d)
Qr = torch.stack([rope(Q[p], torch.tensor(float(p))) for p in range(T)])  # rotate by position
Kr = torch.stack([rope(K[p], torch.tensor(float(p))) for p in range(T)])
attn = F.softmax((Qr @ Kr.T) / d**0.5, dim=-1)
out  = attn @ V
print("attention output shape:", out.shape)            # torch.Size([4, 8])

## Visualize the data & results

_Hold the relative distance m-n fixed and slide both absolute positions up together. RoPE's attention score should stay flat (depends only on m-n); the old ADDITIVE position scheme should drift (depends on absolute m,n). Does it?_

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def rope_2d(v, pos, theta=1.0):
    a = pos * theta
    c, s = np.cos(a), np.sin(a)
    return np.array([c*v[0]-s*v[1], s*v[0]+c*v[1]])

q = np.array([1.0, 0.0]); k = np.array([0.0, 1.0])
# old additive scheme: a fixed pseudo-random position vector per index
P = rng.normal(0, 1, (64, 2))

rope_scores, add_scores = [], []
for m in range(10):
    n = m + 3                                   # relative distance fixed at -3
    # RoPE: rotate each by its own position, then dot
    rope_scores.append(float(rope_2d(q, m) @ rope_2d(k, n)))
    # additive: add a position vector, then dot
    add_scores.append(float((q + P[m]) @ (k + P[n])))

print("RoPE  (flat):", [round(x,4) for x in rope_scores])   # all ~ -0.1411
print("Add   (drift):", [round(x,4) for x in add_scores])   # varies with m
print("RoPE all equal?", max(rope_scores)-min(rope_scores) < 1e-5)  # True

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Rotate the 2-D query $q=[1,2]$ for a token at position $m=3$ with frequency $\theta=0.5$. (Give the rotated vector.)

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Angle: $m\theta = 3\times0.5 = 1.5$ rad. Then $\cos1.5\approx0.0707$, $\sin1.5\approx0.9975$. — _Position times the pair's rotation speed._
- Apply $R(1.5)$: first coord $=\cos\cdot1 - \sin\cdot2 = 0.0707 - 1.995 = -1.9243$. — _Top row is $[\cos,-\sin]$ — the minus sign sits here._
- Second coord $=\sin\cdot1 + \cos\cdot2 = 0.9975 + 0.1414 = 1.139$. — _Bottom row is $[\sin,\cos]$._

**Answer:** $q_3 \approx [-1.9243,\ 1.139]$. Its length is $\sqrt{1^2+2^2}=\sqrt5\approx2.236$, unchanged by the rotation — only the direction moved.

</details>

**Problem 2.** Show, in symbols, why the attention score between a query at position $m$ and a key at position $n$ depends only on $m-n$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write the rotated vectors: query $=R_m q$, key $=R_n k$, with $R_m=R(m\theta)$. — _RoPE rotates each by its own position._
- Dot product: $(R_m q)^\top (R_n k) = q^\top R_m^\top R_n\, k$. — _Move the transpose onto the first matrix: $(Ab)^\top c = b^\top A^\top c$._
- Use $R_m^\top R_n = R_{n-m}$ (rotations are orthogonal, $R(\alpha)^\top=R(-\alpha)$, and rotations add). — _Un-turn by $m$ then turn by $n$ = net turn by $n-m$._

**Answer:** $(R_m q)^\top (R_n k) = q^\top R_{n-m}\, k$ (Eq. 16). The absolute $m$ and $n$ have cancelled; only their difference survives.

</details>

**Problem 3.** Ablation: in the CODE, replace the rotation with the OLD additive scheme — add a position vector $p_m$ to the query instead of rotating it: $q+p_m$, $k+p_n$. Does the score still depend only on $m-n$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Form the additive score: $(q+p_m)^\top(k+p_n) = q^\top k + q^\top p_n + p_m^\top k + p_m^\top p_n$. — _Expand the dot product of the two sums._
- Inspect the cross terms $q^\top p_n$ and $p_m^\top k$: each depends on a single absolute position ($n$ alone, $m$ alone), not on the difference. — _There is no algebraic identity collapsing them to $m-n$._
- Run it: hold $m-n$ fixed but slide $m,n$ up together; the additive score CHANGES, the RoPE score does NOT. — _Only the multiplicative rotation enjoys the $R_m^\top R_n=R_{n-m}$ cancellation._

**Answer:** No. The additive scheme leaves cross terms that depend on absolute $m$ and $n$ separately, so the score drifts as you slide both positions even with $m-n$ fixed. RoPE's multiplicative rotation is exactly what makes the absolute positions cancel — that is the contribution. (Our small run: the additive scores spread across many values while every same-difference RoPE score is identical.)

</details>